# Week 59, Day 4: Comparing models: tests, permutations and uncertainty

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w059/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: McNemar and a paired bootstrap

Write mcnemar(right_a, right_b), taking boolean arrays of which test trials each decoder got right. Then compute a 95% paired bootstrap interval for the difference in accuracy, resampling test trials (the same resample for both decoders).

In [ ]:
import numpy as np
from scipy.stats import chi2, norm
gp = np.random.default_rng(594)

def mcnemar(right_a, right_b):
    b = np.sum(right_a & ~right_b); c = np.sum(~right_a & right_b)
    stat = ...
    return stat, chi2.sf(stat, 1)

# 300 test trials; trial difficulty is shared, and decoder A is slightly better
n = 300
difficulty = gp.normal(size=n)
right_a = (difficulty + 0.62 + 0.3 * gp.normal(size=n)) > 0
right_b = (difficulty + 0.40 + 0.3 * gp.normal(size=n)) > 0
acc_a, acc_b = right_a.mean(), right_b.mean()
stat, p_paired = mcnemar(right_a, right_b)
pooled = (acc_a + acc_b) / 2
z = (acc_a - acc_b) / np.sqrt(2 * pooled * (1 - pooled) / n)
p_unpaired = 2 * norm.sf(abs(z))

boot = []
for _ in range(2000):
    idx = gp.integers(0, n, n)
    boot.append(...)
ci = np.percentile(boot, [2.5, 97.5])
print(f"accuracy A {acc_a:.3f}, B {acc_b:.3f}")
print(f"unpaired two-proportion test p = {p_unpaired:.3f}; McNemar p = {p_paired:.4f}")
print(f"paired bootstrap 95% interval for A minus B: {ci[0]:.3f} to {ci[1]:.3f}")

In [ ]:
import numpy as np
def test_mcnemar():
    "mcnemar returns the chi-squared statistic and its p-value"
    stat, p = mcnemar(np.array([1, 1, 1, 0]), np.array([0, 0, 1, 0]))
    assert abs(stat - 0.5) < 1e-12 and 0.4 < p < 0.5
def test_power():
    "the paired test detects the difference that the unpaired comparison misses"
    assert p_paired < 0.05 < p_unpaired
def test_ci():
    "the bootstrap interval for the accuracy difference excludes zero"
    assert ci[0] > 0

test_mcnemar()
test_power()
test_ci()
print("All checks passed")

### Exercise: A permutation test for a decoder

Complete the permutation test: for 200 shuffles, permute the labels, rerun the 5-fold nearest-centroid cross-validation, and store the accuracy. Compute p = (1 + number of shuffles at least as accurate as the real labels) / (1 + 200).

In [ ]:
def centroid_cv5(X, y, seed=0):
    folds = np.array_split(np.random.default_rng(seed).permutation(len(y)), 5)
    right = 0
    for te in folds:
        tr = np.setdiff1d(np.arange(len(y)), te)
        c0, c1 = X[tr][y[tr] == 0].mean(0), X[tr][y[tr] == 1].mean(0)
        right += np.sum((((X[te] - c1) ** 2).sum(1) < ((X[te] - c0) ** 2).sum(1)) == y[te])
    return right / len(y)

gq = np.random.default_rng(5941)
yq = np.tile([0, 1], 40)
Xq = 0.4 * yq[:, None] * np.ones(15) + gq.normal(size=(80, 15))      # a weak signal in 15 neurons
real = centroid_cv5(Xq, yq)
null = []
for s in range(200):
    null.append(...)
p_perm = ...
print(f"cross-validated accuracy {real:.3f}; null mean {np.mean(null):.3f}, null 95th percentile {np.percentile(null, 95):.3f}; p = {p_perm:.4f}")

In [ ]:
import numpy as np
def test_null():
    "the null distribution has 200 values centred near chance"
    assert len(null) == 200 and abs(np.mean(null) - 0.5) < 0.05
def test_p():
    "the real accuracy is significant, and the p-value counts the observed labels"
    assert p_perm < 0.05 and p_perm >= 1 / 201

test_null()
test_p()
print("All checks passed")

### Exercise: False positives from overlapping splits

For 150 simulated datasets in which decoders A and B (nearest centroid on two different, equally informative sets of neurons) have the same true accuracy, run 15 random 80/20 splits, compute both t-statistics on the 15 paired differences, and record their p-values. Count how often each rejects at 0.05.

In [ ]:
from scipy.stats import t as t_dist

def centroid_acc(Xtr, ytr, Xte, yte):
    c0, c1 = Xtr[ytr == 0].mean(0), Xtr[ytr == 1].mean(0)
    return np.mean((((Xte - c1) ** 2).sum(1) < ((Xte - c0) ** 2).sum(1)) == yte)

gt = np.random.default_rng(5942)
J, n, n_te = 15, 100, 20
p_naive, p_corr = [], []
for dataset in range(150):
    yd = np.tile([0, 1], n // 2)
    Xd = 0.35 * yd[:, None] * np.ones(10) + gt.normal(size=(n, 10))
    diffs = []
    for j in range(J):
        perm = gt.permutation(n); te, tr = perm[:n_te], perm[n_te:]
        a = centroid_acc(Xd[tr][:, :5], yd[tr], Xd[te][:, :5], yd[te])     # neurons 0 to 4
        b = centroid_acc(Xd[tr][:, 5:], yd[tr], Xd[te][:, 5:], yd[te])     # neurons 5 to 9
        diffs.append(a - b)
    d, s2 = np.mean(diffs), np.var(diffs, ddof=1)
    if s2 == 0: s2 = 1e-12
    t_naive = d / np.sqrt(s2 / J)
    t_corr = ...
    p_naive.append(2 * t_dist.sf(abs(t_naive), J - 1)); p_corr.append(2 * t_dist.sf(abs(t_corr), J - 1))
print(f"false positive rate: naive t-test {np.mean(np.array(p_naive) < 0.05):.3f}, corrected {np.mean(np.array(p_corr) < 0.05):.3f} (nominal 0.05)")

In [ ]:
import numpy as np
def test_rates():
    "the naive t-test rejects a true null more than 10% of the time; the corrected test less than 10%"
    assert len(p_naive) == 150 and np.mean(np.array(p_naive) < 0.05) > 0.10 and np.mean(np.array(p_corr) < 0.05) < 0.10

test_rates()
print("All checks passed")